# MLflow 2회차: 여러 실험 비교하고 좋은 모델 저장하기

이 노트북은 MLflow를 처음 배우는 수강생을 위한 **2회차 실습 자료**입니다.

1회차에서는 하나의 모델 실험을 MLflow에 기록하고 UI에서 확인했습니다. 2회차에서는 실험이 여러 개로 늘어났을 때 MLflow를 어떻게 활용하는지 배웁니다.

이번 시간의 목표는 다음과 같습니다.

1. 여러 하이퍼파라미터 조합으로 반복 실험한다.
2. MLflow UI와 DataFrame으로 실험 결과를 비교한다.
3. confusion matrix 같은 결과물을 artifact로 저장한다.
4. RandomForest와 XGBoost 모델을 비교한다.
5. 가장 좋은 모델을 Model Registry에 등록한다.
6. Registry에 저장된 모델을 다시 불러와 예측한다.

> 이 노트북은 1회차 노트북을 실행하지 않아도 단독으로 실행할 수 있도록 구성되어 있습니다.

## 1. 1회차 핵심 복습

2회차를 시작하기 전에 1회차에서 배운 내용을 간단히 복습합니다.

- **Experiment**: 큰 실험 주제
- **Run**: 한 번의 모델 학습 실행
- **Parameter**: 모델 설정값
- **Metric**: 모델 성능값
- **Artifact**: 실험 결과로 저장되는 파일
- **Model Registry**: 좋은 모델을 이름과 버전으로 관리하는 저장소

2회차의 핵심 질문은 다음입니다.

> 실험이 여러 개로 늘어나면, 어떤 실험이 가장 좋았는지 어떻게 찾고 관리할까?

## 2. 기본 코드 준비

2회차 노트북도 단독으로 실행할 수 있도록 데이터 로드, 데이터 분할, 라이브러리 import를 다시 수행합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier

import mlflow

In [ ]:
# 데이터 로드
digits = load_digits()
X = digits.data
y = digits.target

# 학습용/테스트용 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=20,
)

print("학습 데이터 개수:", len(X_train))
print("테스트 데이터 개수:", len(X_test))

In [ ]:
# 1회차와 같은 experiment 이름을 사용합니다.
# 이렇게 하면 1회차와 2회차 실험 결과를 같은 주제 아래에서 볼 수 있습니다.
experiment_name = "digit_classification_test"
mlflow.set_experiment(experiment_name)

## 3. 여러 파라미터 조합으로 반복 실험하기

이번에는 RandomForest의 설정값을 여러 번 바꿔가며 실험해보겠습니다.

초보자에게 중요한 포인트는 다음입니다.

- 파라미터가 달라지면 모델 성능도 달라질 수 있습니다.
- 실험을 여러 번 하면 기록이 많아집니다.
- MLflow를 사용하면 여러 실험 결과를 자동으로 모아서 비교할 수 있습니다.

In [ ]:
# 실험해볼 파라미터 조합을 준비합니다.
# 실제 실무에서는 더 많은 조합을 실험할 수 있지만,
# 수업에서는 이해하기 쉽게 4개 조합만 사용합니다.

rf_param_list = [
    {"n_estimators": 20, "max_depth": 3, "criterion": "gini"},
    {"n_estimators": 50, "max_depth": 4, "criterion": "gini"},
    {"n_estimators": 30, "max_depth": 3, "criterion": "entropy"},
    {"n_estimators": 50, "max_depth": 5, "criterion": "entropy"},
]

for idx, params in enumerate(rf_param_list, start=1):
    with mlflow.start_run(
        run_name=f"RandomForest_{idx}",
        tags={"model": "RandomForestClassifier", "dataset": "digits", "step": "tuning"},
    ):
        # 1. 현재 파라미터로 모델 생성
        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)

        # 2. 예측 및 평가
        pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, pred)
        report = classification_report(y_test, pred, output_dict=True)

        metrics = {
            "accuracy": accuracy,
            "macro_f1_score": report["macro avg"]["f1-score"],
            "weighted_f1_score": report["weighted avg"]["f1-score"],
        }

        # 3. MLflow에 기록
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, name="model")

    print(f"{idx}번째 실험 완료 - accuracy: {accuracy:.4f}")

## 4. 코드로 실험 결과 비교하기

MLflow UI에서도 실험을 비교할 수 있지만, 코드로도 실험 결과를 가져올 수 있습니다.

`mlflow.search_runs()`를 사용하면 특정 Experiment에 속한 Run들을 DataFrame으로 가져옵니다.

이제 정확도 기준으로 어떤 실험이 가장 좋았는지 확인해보겠습니다.

In [ ]:
# 현재 experiment 정보를 가져옵니다.
experiment = mlflow.get_experiment_by_name(experiment_name)

# experiment_id를 기준으로 run 목록을 가져옵니다.
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

runs_df

In [ ]:
columns_to_show = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.macro_f1_score",
    "params.n_estimators",
    "params.max_depth",
    "params.criterion",
    "start_time",
]

runs_df[columns_to_show].sort_values("metrics.accuracy", ascending=False).head(10)

## 5. Artifact 저장하기

지금까지는 숫자 성능 지표만 기록했습니다.

하지만 실험 결과에는 숫자만 있는 것이 아닙니다. 예를 들어 다음과 같은 파일도 함께 저장하면 좋습니다.

- confusion matrix 이미지
- classification report 파일
- feature importance 그래프

MLflow에서는 이런 파일을 **Artifact**라고 부릅니다.

아래에서는 confusion matrix 이미지와 classification report 파일을 MLflow에 저장해보겠습니다.

In [ ]:
# Artifact 저장 예제를 위한 모델을 하나 더 학습합니다.
artifact_params = {
    "n_estimators": 80,
    "max_depth": 6,
    "criterion": "gini",
    "random_state": 20,
}

with mlflow.start_run(
    run_name="RandomForest_with_artifact",
    tags={"model": "RandomForestClassifier", "dataset": "digits", "step": "artifact"},
):
    model = RandomForestClassifier(**artifact_params)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    report = classification_report(y_test, pred, output_dict=True)

    metrics = {
        "accuracy": accuracy,
        "macro_f1_score": report["macro avg"]["f1-score"],
        "weighted_f1_score": report["weighted avg"]["f1-score"],
    }

    mlflow.log_params(artifact_params)
    mlflow.log_metrics(metrics)

    # confusion matrix를 그림으로 만들고 MLflow artifact로 저장합니다.
    fig, ax = plt.subplots(figsize=(7, 7))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, cmap="Blues")
    ax.set_title("Confusion Matrix")
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.close(fig)

    # classification report도 dict 형태로 저장합니다.
    mlflow.log_dict(report, "classification_report.json")

    # 모델도 함께 저장합니다.
    mlflow.sklearn.log_model(model, name="model")

print("Artifact 저장 실험 완료")

## 6. 다른 모델도 기록하기: XGBoost

이번에는 RandomForest가 아닌 XGBoost 모델을 학습하고 MLflow에 기록해보겠습니다.

여기서 중요한 점은 모델 종류에 따라 MLflow의 저장 함수가 달라질 수 있다는 것입니다.

- scikit-learn 모델: `mlflow.sklearn.log_model()`
- XGBoost 모델: `mlflow.xgboost.log_model()`

이런 모델별 저장 방식을 MLflow에서는 **flavor**라고 부릅니다.

In [ ]:
!pip install xgboost

In [ ]:
import mlflow.xgboost
from xgboost import XGBClassifier

xgb_params = {
    "n_estimators": 30,
    "max_depth": 3,
    "learning_rate": 0.3,
    "random_state": 20,
}

with mlflow.start_run(
    run_name="XGBoost_first_run",
    tags={"model": "XGBClassifier", "dataset": "digits"},
):
    model = XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    report = classification_report(y_test, pred, output_dict=True)

    metrics = {
        "accuracy": accuracy,
        "macro_f1_score": report["macro avg"]["f1-score"],
        "weighted_f1_score": report["weighted avg"]["f1-score"],
    }

    mlflow.log_params(xgb_params)
    mlflow.log_metrics(metrics)
    mlflow.xgboost.log_model(model, name="model")

print("XGBoost 실험 기록 완료")

## 7. 가장 좋은 실험 찾기

이제 여러 Run이 쌓였습니다.

다시 `mlflow.search_runs()`로 전체 실험 결과를 가져온 뒤, accuracy가 가장 높은 Run을 찾아보겠습니다.

이 Run의 모델을 Model Registry에 등록할 예정입니다.

In [ ]:
# 최신 run 목록을 다시 가져옵니다.
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# accuracy 기준으로 가장 좋은 run을 찾습니다.
best_run = runs_df.sort_values("metrics.accuracy", ascending=False).iloc[0]

best_run_id = best_run["run_id"]
best_run_name = best_run["tags.mlflow.runName"]
best_accuracy = best_run["metrics.accuracy"]

print("가장 좋은 Run 이름:", best_run_name)
print("가장 좋은 Run ID:", best_run_id)
print("가장 좋은 정확도:", best_accuracy)

## 8. Model Registry에 좋은 모델 등록하기

Run 안에 저장된 모델은 실험 결과물입니다.

그중에서 성능이 좋거나 관리할 가치가 있는 모델은 Model Registry에 등록할 수 있습니다.

초보자 관점에서는 이렇게 이해하면 됩니다.

- Run의 모델: 실험 중 만들어진 모델
- Registry의 모델: 관리 대상으로 선택된 모델

모델을 Registry에 등록하려면 `runs:/run_id/model` 형태의 URI를 사용합니다.

In [ ]:
# best_run_id에 해당하는 run 안의 model artifact를 가리킵니다.
best_model_uri = f"runs:/{best_run_id}/model"

# Registry에 등록할 모델 이름입니다.
# 같은 이름으로 여러 번 등록하면 version이 1, 2, 3처럼 증가합니다.
registered_model_name = "Beginner_Digit_Classifier"

registered_model = mlflow.register_model(
    model_uri=best_model_uri,
    name=registered_model_name,
    tags={"task": "digit_classification", "level": "beginner"},
)

registered_model

## 9. Registry에 저장된 모델 다시 불러오기

마지막으로 Registry에 등록된 모델을 다시 불러와 예측해보겠습니다.

모델을 불러올 때는 다음 형태의 URI를 사용합니다.

```text
models:/모델이름/버전
```

방금 등록한 모델의 버전을 사용해서 모델을 로드해보겠습니다.

In [ ]:
# 등록된 모델의 버전을 가져옵니다.
model_version = registered_model.version

# Registry 모델 URI를 만듭니다.
model_uri = f"models:/{registered_model_name}/{model_version}"
print("불러올 모델 URI:", model_uri)

# 모델 flavor와 상관없이 pyfunc로 불러오면 predict 인터페이스를 공통으로 사용할 수 있습니다.
# 초보자에게는 '저장된 모델을 다시 불러오는 공통 방식'으로 이해하면 됩니다.
loaded_model = mlflow.pyfunc.load_model(model_uri)

# 불러온 모델로 예측합니다.
loaded_pred = loaded_model.predict(X_test)

loaded_accuracy = accuracy_score(y_test, loaded_pred)
print("불러온 모델의 정확도:", loaded_accuracy)

In [ ]:
# 불러온 모델의 상세 성능도 확인해봅니다.
loaded_report = classification_report(y_test, loaded_pred, output_dict=True)
pd.DataFrame(loaded_report).T